In [1]:
import chromadb
import pandas as pd
import os

# 1. Configuration
db_path = os.path.abspath(r"D:\pY\InsuranceRAG\local_db")
print(f"📂 Looking for database at: {db_path}")

client = chromadb.PersistentClient(path=db_path)
collections = client.list_collections()
collection_names = [c.name for c in collections]
print(f"📚 Found {len(collections)} collections: {collection_names}")

# 2. Function to inspect a specific collection
def inspect_collection_dynamic(name):
    if name not in collection_names:
        print(f"❌ Collection '{name}' does not exist.")
        return None
    
    collection = client.get_collection(name=name)
    results = collection.get(include=["documents", "metadatas"])
    
    if not results['ids']:
        print(f"⚠️ Collection '{name}' exists but is empty.")
        return None
        
    records = []
    for i in range(len(results['ids'])):
        # 1. Start with the core fields (ID and Document Preview)
        entry = {
            "ID": results['ids'][i],
            "Content Preview": results['documents'][i][:100].replace('\n', ' ') + "..."
        }
        
        # 2. Dynamically add all metadata fields
        meta = results['metadatas'][i] if results['metadatas'] else {}
        for key, value in meta.items():
            # If the value is a file path, just show the filename for neatness
            if key == "source" and isinstance(value, str):
                entry["Source File"] = os.path.basename(value)
            else:
                entry[key] = value
                
        records.append(entry)
    
    # 3. Create DataFrame (Pandas handles the alignment of different keys automatically)
    df = pd.DataFrame(records)
    
    # Optional: Move 'ID' and 'Content Preview' to the end for better readability
    cols = [c for c in df.columns if c not in ["ID", "Content Preview"]] + ["ID", "Content Preview"]
    return df[cols]

# 3. Fetch and Display Both Collections
print("\n--- 📜 MASTER POLICIES ---")
df_policies = inspect_collection_dynamic("policy_master_collection")
if df_policies is not None:
    display(df_policies.sort_values(by="Source File"))

print("\n--- 📑 CLIENT CLAIMS ---")
df_claims = inspect_collection_dynamic("claims_collection")
if df_claims is not None:
    display(df_claims)

print("\n--- 📑 EVALUATION AUDIT LOG ---")
df_audit = inspect_collection_dynamic("evaluation_audit_log")
if df_audit is not None:
    display(df_audit)

print("\n--- 📑 INSTANCE LOG ---")
df_instance_log = inspect_collection_dynamic("instance_log")
if df_instance_log is not None:
    display(df_instance_log)

# 4. Summary Table
if df_policies is not None or df_claims is not None:
    print("\n📊 Database Summary:")
    for c in collections:
        print(f" - {c.name}: {c.count()} chunks")

📂 Looking for database at: D:\pY\InsuranceRAG\local_db
📚 Found 4 collections: ['claims_collection', 'evaluation_audit_log', 'instance_log', 'policy_master_collection']

--- 📜 MASTER POLICIES ---


,moddate,page,document_category,Source File,creationDate,creator,modDate,creationdate,producer,file_path,title,format,keywords,author,total_pages,subject,trapped,source_type,ID,Content Preview
154,2022-07-07T17:33:44+04:00,17,Helath,Membership Handbook.pdf,D:20220707173342+04'00',Adobe InDesign 15.1 (Macintosh),D:20220707173344+04'00',2022-07-07T17:33:42+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Helath\202...,,PDF 1.4,,,18,,,Policy,620cb565-fc58-4acd-9a74-90d28904e4ae,(6) of 2007 concerning the establishment of In...
128,2022-07-07T17:33:44+04:00,7,Helath,Membership Handbook.pdf,D:20220707173342+04'00',Adobe InDesign 15.1 (Macintosh),D:20220707173344+04'00',2022-07-07T17:33:42+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Helath\202...,,PDF 1.4,,,18,,,Policy,3546c24a-9c78-4631-af31-77df794614c2,When we use these terms they are in bold prin...
127,2022-07-07T17:33:44+04:00,7,Helath,Membership Handbook.pdf,D:20220707173342+04'00',Adobe InDesign 15.1 (Macintosh),D:20220707173344+04'00',2022-07-07T17:33:42+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Helath\202...,,PDF 1.4,,,18,,,Policy,251f04ce-e865-4cc6-8309-2e73da7e4862,Policy Handbook Health Insurance 14 15 to be ...
126,2022-07-07T17:33:44+04:00,6,Helath,Membership Handbook.pdf,D:20220707173342+04'00',Adobe InDesign 15.1 (Macintosh),D:20220707173344+04'00',2022-07-07T17:33:42+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Helath\202...,,PDF 1.4,,,18,,,Policy,f0dc7cdd-6f6c-46cd-b850-94d985d97ec8,Please also note that no claim of any kind wi...
125,2022-07-07T17:33:44+04:00,6,Helath,Membership Handbook.pdf,D:20220707173342+04'00',Adobe InDesign 15.1 (Macintosh),D:20220707173344+04'00',2022-07-07T17:33:42+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Helath\202...,,PDF 1.4,,,18,,,Policy,9987c52a-8572-4306-a043-81e16a84abd6,What happens if you wish to cancel your poli...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,2022-06-16T13:34:20+04:00,10,Travel,TRAVEL_TC_HSBCUAE_EN.pdf,D:20220616133418+04'00',Adobe InDesign 15.1 (Macintosh),D:20220616133420+04'00',2022-06-16T13:34:18+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Travel\202...,,PDF 1.4,,,18,,,Policy,cf669d4a-e4b2-432e-b8a6-84b37fad71c1,What is not covered a)\t Your failure to check...
26,2022-06-16T13:34:20+04:00,10,Travel,TRAVEL_TC_HSBCUAE_EN.pdf,D:20220616133418+04'00',Adobe InDesign 15.1 (Macintosh),D:20220616133420+04'00',2022-06-16T13:34:18+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Travel\202...,,PDF 1.4,,,18,,,Policy,6112f350-86ab-4b5b-a951-4bbecba64146,Hijack of the Insured Person or of any person...
25,2022-06-16T13:34:20+04:00,10,Travel,TRAVEL_TC_HSBCUAE_EN.pdf,D:20220616133418+04'00',Adobe InDesign 15.1 (Macintosh),D:20220616133420+04'00',2022-06-16T13:34:18+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Travel\202...,,PDF 1.4,,,18,,,Policy,7781282d-7a74-4ba7-a287-c4c9c7dc45db,Policy Handbook Travel Smart 20 21 bail bond ...
34,2022-06-16T13:34:20+04:00,13,Travel,TRAVEL_TC_HSBCUAE_EN.pdf,D:20220616133418+04'00',Adobe InDesign 15.1 (Macintosh),D:20220616133420+04'00',2022-06-16T13:34:18+04:00,Adobe PDF Library 15.0,D:\pY\InsuranceRAG\storage\policies\Travel\202...,,PDF 1.4,,,18,,,Policy,7115c50e-730b-4c87-a954-eaa12f69126d,Policy Handbook Travel Smart 26 27 8 Gener...



--- 📑 CLIENT CLAIMS ---
⚠️ Collection 'claims_collection' exists but is empty.

--- 📑 EVALUATION AUDIT LOG ---


,instance_id,status,risk_score,ID,Content Preview
0,f1cfd046-8dcb-4531-ab3f-85a107324240,ERROR,NaN,df7f86b4-4a04-4c35-a105-de4f865fade8,Critical Graph Failure: unsupported operand ty...
1,cce39e15-d794-4767-a99b-b1594cbf6378,ERROR,NaN,693fac93-887a-43e8-9b18-d506bb251038,Critical Graph Failure: unsupported operand ty...
2,cc11f4dc-95b9-4c58-869e-5c6c649ea89e,ERROR,NaN,214b93af-1b3d-4595-8332-1e585aa37603,Critical Graph Failure: unsupported operand ty...
3,9cb41797-a978-4846-be14-8d77b65da41a,ERROR,NaN,8d89043b-99c3-4879-acd3-638a1d47c9d5,Critical Graph Failure: unsupported operand ty...
4,b03981a8-06d6-4158-8e7b-297337603a8e,ERROR,NaN,5eef19d0-0c48-46a7-a7e1-372c5f707b52,Critical Graph Failure: unsupported operand ty...
5,a1cbd652-337a-4db5-91d5-a7d606404369,ERROR,NaN,c082b6a9-e1e7-436b-935c-0691678a0842,Critical Graph Failure: unsupported operand ty...
6,92d229da-6a41-4aea-ab57-f269ef236dd1,ERROR,NaN,2e27540b-950f-463f-aed5-ec21d9c08676,Critical Graph Failure: unsupported operand ty...
7,f969116a-c180-4b1d-a769-98089d6f713e,ERROR,NaN,9418175e-e45d-4fdc-85c4-9842d5c812ef,Critical Graph Failure: unsupported operand ty...
8,28f4ad93-557f-49a8-ad65-52af0c480b94,ERROR,NaN,b9125fbe-0fad-40bd-a754-bf4c3b2e1f59,Critical Graph Failure: unsupported operand ty...
9,18d284b0-64e0-4f49-9b33-eb813be101f5,Policy Selection_running,NaN,5c517e13-1637-4db0-a11a-95fbf300791b,Started Policy Selection...



--- 📑 INSTANCE LOG ---


,error_details,client_id,instance_id,status,completion_time,risk_score,ID,Content Preview
0,unsupported operand type(s) for +: 'dict' and ...,12345,f1cfd046-8dcb-4531-ab3f-85a107324240,FAILED,2026-02-11T02:02:08.118007,NaN,f1cfd046-8dcb-4531-ab3f-85a107324240,Investigation terminated: unsupported operand ...
1,unsupported operand type(s) for +: 'dict' and ...,12345,cce39e15-d794-4767-a99b-b1594cbf6378,FAILED,2026-02-11T02:09:01.635914,NaN,cce39e15-d794-4767-a99b-b1594cbf6378,Investigation terminated: unsupported operand ...
2,unsupported operand type(s) for +: 'dict' and ...,12345,cc11f4dc-95b9-4c58-869e-5c6c649ea89e,FAILED,2026-02-11T02:12:47.266911,NaN,cc11f4dc-95b9-4c58-869e-5c6c649ea89e,Investigation terminated: unsupported operand ...
3,unsupported operand type(s) for +: 'dict' and ...,12345,9cb41797-a978-4846-be14-8d77b65da41a,FAILED,2026-02-11T02:15:45.876219,NaN,9cb41797-a978-4846-be14-8d77b65da41a,Investigation terminated: unsupported operand ...
4,unsupported operand type(s) for +: 'dict' and ...,12345,b03981a8-06d6-4158-8e7b-297337603a8e,FAILED,2026-02-11T02:18:54.524544,NaN,b03981a8-06d6-4158-8e7b-297337603a8e,Investigation terminated: unsupported operand ...
5,unsupported operand type(s) for +: 'dict' and ...,12345,a1cbd652-337a-4db5-91d5-a7d606404369,FAILED,2026-02-11T02:22:36.567189,NaN,a1cbd652-337a-4db5-91d5-a7d606404369,Investigation terminated: unsupported operand ...
6,unsupported operand type(s) for +: 'dict' and ...,12345,92d229da-6a41-4aea-ab57-f269ef236dd1,FAILED,2026-02-11T02:26:43.056154,NaN,92d229da-6a41-4aea-ab57-f269ef236dd1,Investigation terminated: unsupported operand ...
7,unsupported operand type(s) for +: 'dict' and ...,12345,f969116a-c180-4b1d-a769-98089d6f713e,FAILED,2026-02-11T02:30:00.763836,NaN,f969116a-c180-4b1d-a769-98089d6f713e,Investigation terminated: unsupported operand ...
8,unsupported operand type(s) for +: 'dict' and ...,12345,28f4ad93-557f-49a8-ad65-52af0c480b94,FAILED,2026-02-11T02:34:25.955033,NaN,28f4ad93-557f-49a8-ad65-52af0c480b94,Investigation terminated: unsupported operand ...
9,[WinError 10061] No connection could be made b...,12345,18d284b0-64e0-4f49-9b33-eb813be101f5,FAILED,2026-02-11T02:38:20.262073,NaN,18d284b0-64e0-4f49-9b33-eb813be101f5,Investigation terminated: [WinError 10061] No ...



📊 Database Summary:
 - claims_collection: 0 chunks
 - evaluation_audit_log: 35 chunks
 - instance_log: 12 chunks
 - policy_master_collection: 155 chunks
